# Stage 4 - Cache model outputs

This notebook caches predictions from the fixed standard and AugMix CIFAR-style ResNet-18 checkpoints. It does not train, select checkpoints, calibrate, or build conformal sets.

For each model, it performs one full 10,000-image pass on clean CIFAR-10 and one pass for each approved CIFAR-10-C condition: Gaussian noise, defocus blur, and brightness at severities 1, 3, and 5. The same caches are indexed later by split seeds 42, 43, and 44; inference is not repeated per split.

**Index rule:** severity blocks in CIFAR-10-C have row offsets, but the canonical original CIFAR-10 test index is always 0-9999. The cache stores `original_test_indices` separately from `cifar10c_source_rows` and checks labels against clean test labels.

**Outputs:** one `.npz` cache per model-condition, model metadata, a manifest, and a top-1 accuracy CSV under `cached_outputs/stage4_predictions/`.


## 1. Project and CIFAR-10-C data

The first code cell uses Python's standard library plus NumPy, so you can run it in local Jupyter to download and extract the data without importing PyTorch. The official archive is about 2.9 GB; the three selected corruption arrays plus labels take about 461 MB after extraction. The local files go under this project's `data/raw/cifar-10-c-selected/` on E:.

For the later model inference, use Colab because the local Python environment currently cannot import PyTorch. The simplest route is to run this same first code cell in Colab: it stores the archive in temporary runtime space and extracts only the selected arrays into the Drive project. If you download locally, upload the extracted `cifar-10-c-selected/` folder into `MyDrive/conformal_shift/data/raw/` before running the cache cells in Colab.

The official archive includes 15 corruptions. This notebook extracts only the three project-approved arrays and labels. Existing arrays are reused, so a second run will not download them again.


Saved cell outputs below document the original verified Stage 4 cache-generation run. On a rerun with the complete cache set, the notebook performs a cache-only audit and keeps the existing files unchanged.


In [ ]:

import hashlib, json, os, shutil, sys, tarfile, urllib.request
from pathlib import Path
import numpy as np

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/conformal_shift")
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_ROOT = PROJECT_ROOT / "data" / "raw"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
CIFAR_C_DIR = DATA_ROOT / "cifar-10-c-selected"
CIFAR_C_TAR = Path("/content/CIFAR-10-C.tar") if IN_COLAB else DATA_ROOT / "CIFAR-10-C.tar"
OUTPUT_ROOT = PROJECT_ROOT / "cached_outputs" / "stage4_predictions"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Running this cell downloads the official 2.9 GB tar only if selected arrays are absent.
DOWNLOAD_OFFICIAL_CIFAR_C = True
ARCHIVE_URL = "https://zenodo.org/records/2535967/files/CIFAR-10-C.tar?download=1"
ARCHIVE_BYTES = 2_918_471_680
ARCHIVE_MD5 = "56bf5dcef84df0e2308c6dcbcbbd8499"
SELECTED_FILES = ["gaussian_noise.npy", "defocus_blur.npy", "brightness.npy", "labels.npy"]
CACHE_MODEL_KEYS = ("standard_resnet18", "augmix_resnet18")
CACHE_CONDITIONS = ["clean_test.npz"] + [
    f"{corruption}_severity_{severity}.npz"
    for corruption in ("gaussian_noise", "defocus_blur", "brightness")
    for severity in (1, 3, 5)
]
EXPECTED_CACHE_FILES = {
    f"{model_key}/{filename}"
    for model_key in CACHE_MODEL_KEYS
    for filename in CACHE_CONDITIONS
}
manifest_path = OUTPUT_ROOT / "cache_manifest.json"
accuracy_path = OUTPUT_ROOT / "cache_accuracy_summary.csv"
metadata_paths = [OUTPUT_ROOT / key / "model_metadata.json" for key in CACHE_MODEL_KEYS]
try:
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    listed_cache_files = set(existing_manifest.get("cache_files", []))
    CACHE_REUSE_ONLY = (
        existing_manifest.get("stage") == "stage4_model_output_cache"
        and listed_cache_files == EXPECTED_CACHE_FILES
        and len(listed_cache_files) == 20
        and all((OUTPUT_ROOT / name).is_file() for name in listed_cache_files)
        and accuracy_path.is_file()
        and all(path.is_file() for path in metadata_paths)
    )
except (OSError, ValueError, TypeError):
    CACHE_REUSE_ONLY = False

if CACHE_REUSE_ONLY:
    print("Complete Stage 4 cache set found: cache-only validation; no CIFAR-10-C download or inference.")
else:
    print("Complete cache set not found: cache generation may be needed.")


def md5_file(path):
    digest = hashlib.md5()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def download_archive(path):
    path = Path(path)
    partial = path.with_suffix(path.suffix + ".partial")
    path.parent.mkdir(parents=True, exist_ok=True)
    received = 0
    reported = 0
    with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response, partial.open("wb") as out:
        size = response.headers.get("Content-Length")
        if size and int(size) != ARCHIVE_BYTES:
            raise ValueError(f"Unexpected archive size: {size}")
        while True:
            block = response.read(8 * 1024 * 1024)
            if not block:
                break
            out.write(block)
            received += len(block)
            if received - reported >= 512 * 1024 * 1024:
                print(f"Downloaded {received / 1024**3:.2f} / {ARCHIVE_BYTES / 1024**3:.2f} GiB")
                reported = received
    if received != ARCHIVE_BYTES or md5_file(partial) != ARCHIVE_MD5:
        raise ValueError("CIFAR-10-C archive size or MD5 verification failed.")
    partial.replace(path)
    print(f"Verified official CIFAR-10-C archive: {path}")


def extract_selected(path, destination):
    wanted = set(SELECTED_FILES)
    destination = Path(destination)
    found = {}
    with tarfile.open(path, mode="r:") as archive:
        for member in archive:
            base = Path(member.name).name
            if member.isfile() and base in wanted:
                if base in found:
                    raise ValueError(f"Duplicate archive member: {base}")
                found[base] = member
        if set(found) != wanted:
            raise FileNotFoundError(f"Selected CIFAR-10-C members missing: {sorted(wanted - set(found))}")
        for name, member in found.items():
            target = destination / name
            if target.exists():
                print(f"Keeping existing file: {target}")
                continue
            source = archive.extractfile(member)
            if source is None:
                raise OSError(f"Could not read archive member: {member.name}")
            with source, target.open("wb") as out:
                shutil.copyfileobj(source, out, length=8 * 1024 * 1024)
            print(f"Extracted {target.name}")

if CACHE_REUSE_ONLY:
    print("CIFAR-10-C source arrays are not needed because all 20 caches already exist.")
else:
    CIFAR_C_DIR.mkdir(parents=True, exist_ok=True)
    missing = [name for name in SELECTED_FILES if not (CIFAR_C_DIR / name).is_file()]
    if missing:
        if not CIFAR_C_TAR.is_file():
            if not DOWNLOAD_OFFICIAL_CIFAR_C:
                raise FileNotFoundError(f"Missing {missing}; provide the selected arrays or enable official download.")
            download_archive(CIFAR_C_TAR)
        if md5_file(CIFAR_C_TAR) != ARCHIVE_MD5:
            raise ValueError("The CIFAR-10-C tar MD5 does not match the official archive.")
        extract_selected(CIFAR_C_TAR, CIFAR_C_DIR)

    missing = [name for name in SELECTED_FILES if not (CIFAR_C_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Selected CIFAR-10-C arrays remain missing: {missing}")
    print(f"Selected CIFAR-10-C arrays are ready: {CIFAR_C_DIR}")



## 2. Runtime, deterministic transforms, and fixed models

Clean and corrupted images use only `ToTensor()` and the training CIFAR-10 normalization. There is no random crop or flip. Both models run in `eval()` with `torch.inference_mode()`.


In [ ]:

import gc, json, platform, time, zipfile
import pandas as pd
import torch, torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from IPython.display import display

print(f"Python: {platform.python_version()} | PyTorch: {torch.__version__} | torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    print(f"Device: {torch.cuda.get_device_name(DEVICE)} ({torch.cuda.get_device_properties(DEVICE).total_memory/1024**3:.2f} GiB)")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu:0")
    print(f"Device: {torch.xpu.get_device_name(DEVICE)}")
else:
    DEVICE = torch.device("cpu")
    print("Device: CPU")
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
EVAL_TRANSFORM = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
N_TEST = 10_000
BATCH_SIZE = 128
NUM_WORKERS = 2
PIN_MEMORY = DEVICE.type == "cuda"
CORRUPTIONS = {"gaussian_noise":"gaussian_noise.npy", "defocus_blur":"defocus_blur.npy", "brightness":"brightness.npy"}
SEVERITIES = (1, 3, 5)

cifar_dir = DATA_ROOT / "cifar-10-batches-py"
original_indices = np.arange(N_TEST, dtype=np.int64)
if CACHE_REUSE_ONLY:
    clean_images = None
    clean_cache_path = OUTPUT_ROOT / "standard_resnet18" / "clean_test.npz"
    with np.load(clean_cache_path, allow_pickle=False) as cached_clean:
        clean_labels = np.asarray(cached_clean["labels"], dtype=np.int64)
        clean_indices = np.asarray(cached_clean["original_test_indices"], dtype=np.int64)
    if clean_labels.shape != (N_TEST,) or not np.array_equal(clean_indices, original_indices):
        raise ValueError("The clean cache cannot provide valid canonical labels for cache-only audit.")
    print("Using canonical labels from the clean cache; raw image files are not needed.")
else:
    if not (cifar_dir / "test_batch").is_file():
        zip_candidates = []
        for search_root in (PROJECT_ROOT, DATA_ROOT, PROJECT_ROOT / "data" / "archives"):
            if search_root.is_dir():
                zip_candidates.extend(search_root.glob("cifar-10-batches-py*.zip"))
                zip_candidates.extend(search_root.glob("cifar10_batches_user_download.zip"))
        zip_candidates = sorted({path.resolve() for path in zip_candidates})
        if not zip_candidates:
            raise FileNotFoundError(
                f"CIFAR-10 test batch is missing at {cifar_dir}. Place the existing ZIP under data/archives "
                "or extract it under data/raw/cifar-10-batches-py."
            )
        cifar_zip = zip_candidates[0]
        print(f"Extracting the existing CIFAR-10 archive: {cifar_zip.name}")
        with zipfile.ZipFile(cifar_zip) as archive:
            safe_root = DATA_ROOT.resolve()
            for member in archive.infolist():
                target = (DATA_ROOT / member.filename).resolve()
                if target != safe_root and safe_root not in target.parents:
                    raise ValueError(f"Unsafe path in CIFAR-10 ZIP: {member.filename}")
            archive.extractall(DATA_ROOT)
        if not (cifar_dir / "test_batch").is_file():
            raise FileNotFoundError(f"The CIFAR-10 ZIP did not contain {cifar_dir / 'test_batch'}")
    clean_test = CIFAR10(root=str(DATA_ROOT), train=False, download=False, transform=None)
    clean_images = clean_test.data
    clean_labels = np.asarray(clean_test.targets, dtype=np.int64)
    if clean_images.shape != (N_TEST,32,32,3) or clean_labels.shape != (N_TEST,):
        raise ValueError("Unexpected clean CIFAR-10 test shape.")

for seed in (42,43,44):
    path = SPLIT_DIR / f"clean_test_split_seed{seed}.npz"
    with np.load(path, allow_pickle=False) as z:
        cal, ev = z["calibration_indices"], z["evaluation_indices"]
    combined = np.r_[cal, ev]
    if (len(cal)!=5000 or len(ev)!=5000 or np.intersect1d(cal,ev).size
            or np.any(combined < 0) or np.any(combined >= N_TEST)
            or not np.array_equal(np.sort(combined), original_indices)):
        raise ValueError(f"Split {seed} failed size, bounds, disjointness, or coverage checks.")
    print(f"Split {seed}: disjoint 5,000/5,000: PASS")

MODEL_SPECS = {
    "standard_resnet18": ("Standard ResNet-18", PROJECT_ROOT/"checkpoints/resnet18_standard_colab/final_model.pt", PROJECT_ROOT/"checkpoints/resnet18_standard_colab/training_config.json"),
    "augmix_resnet18": ("AugMix ResNet-18", PROJECT_ROOT/"checkpoints/resnet18_augmix_colab/final_model.pt", PROJECT_ROOT/"checkpoints/resnet18_augmix_colab/training_config.json"),
}

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024), b""):
            h.update(block)
    return h.hexdigest()

MODEL_META = {}
if CACHE_REUSE_ONLY:
    for key, (display, _, _) in MODEL_SPECS.items():
        metadata_path = OUTPUT_ROOT / key / "model_metadata.json"
        if not metadata_path.is_file():
            raise FileNotFoundError(f"Missing cached model metadata for {key}: {metadata_path}")
        MODEL_META[key] = json.loads(metadata_path.read_text(encoding="utf-8"))
        if MODEL_META[key].get("model_key") != key or MODEL_META[key].get("display_name") != display:
            raise ValueError(f"Cached model metadata does not match {key}.")
        checkpoint_hash = MODEL_META[key].get("checkpoint_sha256", "")
        if len(checkpoint_hash) != 64:
            raise ValueError(f"Cached checkpoint hash is invalid for {key}.")
        if existing_manifest.get("models", {}).get(key) != MODEL_META[key]:
            raise ValueError(f"Manifest/model metadata mismatch for {key}.")
    print("Loaded model/checkpoint provenance from cache metadata sidecars.")
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.models import resnet18_cifar
    for key, (display, checkpoint, config_path) in MODEL_SPECS.items():
        if not checkpoint.is_file() or not config_path.is_file():
            raise FileNotFoundError(f"Missing fixed checkpoint/config for {key}.")
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if config.get("model")!="CIFAR-style ResNet-18" or config.get("num_classes")!=10:
            raise ValueError(f"Unexpected architecture config for {key}.")
        if config.get("normalization_mean")!=list(CIFAR_MEAN) or config.get("normalization_std")!=list(CIFAR_STD):
            raise ValueError(f"Training/evaluation normalization mismatch for {key}.")
        MODEL_META[key] = {"model_key":key,"display_name":display,"checkpoint_file":str(checkpoint.relative_to(PROJECT_ROOT)),"checkpoint_sha256":sha256_file(checkpoint),"training_config_file":str(config_path.relative_to(PROJECT_ROOT)),"training_config_sha256":sha256_file(config_path),"architecture":config["model"],"normalization_mean":list(CIFAR_MEAN),"normalization_std":list(CIFAR_STD),"device":str(DEVICE)}
        print(f"{display}: fixed checkpoint/config verified; SHA-256 {MODEL_META[key]['checkpoint_sha256']}")



## 3. Verify CIFAR-10-C label alignment

CIFAR-10-C stores five consecutive 10,000-image severity blocks. For severity `s`, source rows are `[(s-1)*10000, s*10000)`. The canonical original index is always `0..9999`; source rows are recorded separately. Each selected label block must exactly match clean labels before inference starts.


In [5]:

C_IMAGES = {}
if CACHE_REUSE_ONLY:
    C_LABELS = None
    print("Skipping CIFAR-10-C array loading; cached labels will be checked against the clean cache.")
else:
    C_LABELS = np.load(CIFAR_C_DIR/"labels.npy", mmap_mode="r", allow_pickle=False)
    if C_LABELS.shape!=(50_000,) or not np.issubdtype(C_LABELS.dtype,np.integer):
        raise ValueError(f"Unexpected labels.npy shape/dtype: {C_LABELS.shape}, {C_LABELS.dtype}")
    for name, filename in CORRUPTIONS.items():
        arr = np.load(CIFAR_C_DIR/filename, mmap_mode="r", allow_pickle=False)
        if arr.shape!=(50_000,32,32,3) or arr.dtype!=np.uint8:
            raise ValueError(f"Unexpected {filename} shape/dtype: {arr.shape}, {arr.dtype}")
        C_IMAGES[name] = arr

    for severity in SEVERITIES:
        lo, hi = (severity-1)*N_TEST, severity*N_TEST
        if not np.array_equal(np.asarray(C_LABELS[lo:hi],dtype=np.int64), clean_labels):
            raise ValueError(f"Label mismatch for severity {severity}; stop before inference.")
        print(f"Severity {severity}: all 10,000 labels align by canonical index: PASS")

class IndexedImages(Dataset):
    def __init__(self, images, labels, source_rows):
        self.images, self.labels = images, np.asarray(labels,dtype=np.int64)
        self.source_rows = np.asarray(source_rows,dtype=np.int64)
        if not (len(images)==len(self.labels)==len(self.source_rows)==N_TEST):
            raise ValueError("Image, label, or source-row length is not 10,000.")
    def __len__(self): return N_TEST
    def __getitem__(self, i):
        image = Image.fromarray(np.array(self.images[i],dtype=np.uint8,copy=True))
        return EVAL_TRANSFORM(image), int(self.labels[i]), int(original_indices[i]), int(self.source_rows[i])

CLEAN_SOURCE_ROWS = np.full(N_TEST,-1,dtype=np.int64)
CONDITIONS = [("clean",None)] + [(name,s) for name in CORRUPTIONS for s in SEVERITIES]

def condition_data(name, severity):
    if name=="clean":
        return clean_images, clean_labels, CLEAN_SOURCE_ROWS
    lo,hi=(severity-1)*N_TEST,severity*N_TEST
    if CACHE_REUSE_ONLY:
        return None, clean_labels, np.arange(lo, hi, dtype=np.int64)
    labels=np.asarray(C_LABELS[lo:hi],dtype=np.int64)
    if not np.array_equal(labels,clean_labels):
        raise ValueError(f"Index/label mismatch: {name}, severity {severity}.")
    return C_IMAGES[name][lo:hi],labels,np.arange(lo,hi,dtype=np.int64)


Severity 1: all 10,000 labels align by canonical index: PASS
Severity 3: all 10,000 labels align by canonical index: PASS
Severity 5: all 10,000 labels align by canonical index: PASS



## 4. Cache validation and one-pass inference

Each `.npz` contains logits/probabilities `(10000,10)`, labels, canonical indices, separate CIFAR-10-C source rows, and model/checkpoint/condition metadata. Existing valid caches are reused; an invalid or mismatched cache stops instead of being overwritten.


In [7]:

def validate_cache(a, expected_meta, labels, source_rows):
    logits, probs = np.asarray(a["logits"]), np.asarray(a["probabilities"])
    y, idx, rows = a["labels"].astype(np.int64), a["original_test_indices"].astype(np.int64), a["cifar10c_source_rows"].astype(np.int64)
    if logits.shape!=(N_TEST,10) or probs.shape!=(N_TEST,10): raise ValueError(f"Bad output shapes: {logits.shape}, {probs.shape}")
    if not (len(idx)==len(y)==len(rows)==N_TEST): raise ValueError("Index/label/source-row length mismatch.")
    if not np.array_equal(idx,original_indices): raise ValueError("Canonical original indices are not exactly 0..9999.")
    if not np.array_equal(y,labels) or not np.array_equal(rows,source_rows): raise ValueError("Cached labels or CIFAR-10-C source rows do not align.")
    if not np.isfinite(logits).all() or not np.isfinite(probs).all(): raise ValueError("NaN/Inf in cache.")
    if np.any((probs<0)|(probs>1)) or not np.allclose(probs.sum(1),1,atol=1e-5,rtol=1e-5): raise ValueError("Invalid probability values or row sums.")
    if not np.array_equal(logits.argmax(1),probs.argmax(1)): raise ValueError("Logit/probability argmax mismatch.")
    meta=json.loads(str(a["metadata_json"].item()))
    metadata_keys = (
        "model_key", "display_name", "checkpoint_file", "checkpoint_sha256",
        "training_config_file", "training_config_sha256", "architecture",
        "normalization_mean", "normalization_std", "condition", "severity",
        "n_images", "canonical_original_test_indices", "cifar10c_source_row_range",
        "transform", "model_eval", "inference_mode",
    )
    for k in metadata_keys:
        if meta.get(k)!=expected_meta.get(k): raise ValueError(f"Cache metadata mismatch: {k}")
    return logits, probs, y, meta

def cache_one(model_key, model, name, severity):
    images, labels, source_rows = condition_data(name,severity)
    display, checkpoint, config_path = MODEL_SPECS[model_key]
    base = {**MODEL_META[model_key],"condition":name,"severity":severity,"n_images":N_TEST,"canonical_original_test_indices":[0,N_TEST-1],"cifar10c_source_row_range":None if name=="clean" else [int(source_rows[0]),int(source_rows[-1])+1],"transform":["ToTensor",f"Normalize(mean={CIFAR_MEAN}, std={CIFAR_STD})"],"model_eval":True,"inference_mode":True}
    folder=OUTPUT_ROOT/model_key
    folder.mkdir(parents=True,exist_ok=True)
    fname="clean_test.npz" if name=="clean" else f"{name}_severity_{severity}.npz"
    path=folder/fname
    if path.exists():
        with np.load(path,allow_pickle=False) as z: a={k:z[k] for k in z.files}
        logits,probs,y,_=validate_cache(a,base,labels,source_rows)
        print(f"Reusing verified cache: {path.name}")
    else:
        if CACHE_REUSE_ONLY:
            raise FileNotFoundError(f"Cache-only mode found a missing cache and will not run inference: {path}")
        ds=IndexedImages(images,labels,source_rows)
        loader=DataLoader(ds,batch_size=128,shuffle=False,num_workers=2,pin_memory=DEVICE.type=="cuda")
        ls,ps,ys,is_,rs=[],[],[],[],[]
        model.eval()
        with torch.inference_mode():
            for x,b_y,b_i,b_r in loader:
                out=model(x.to(DEVICE,non_blocking=DEVICE.type=="cuda")).float()
                ls.append(out.cpu().numpy()); ps.append(torch.softmax(out,dim=1).cpu().numpy())
                ys.append(b_y.numpy()); is_.append(b_i.numpy()); rs.append(b_r.numpy())
        a={"logits":np.concatenate(ls).astype(np.float32),"probabilities":np.concatenate(ps).astype(np.float32),"labels":np.concatenate(ys).astype(np.int64),"original_test_indices":np.concatenate(is_).astype(np.int64),"cifar10c_source_rows":np.concatenate(rs).astype(np.int64),"metadata_json":np.asarray(json.dumps(base,sort_keys=True))}
        validate_cache(a,base,labels,source_rows)
        tmp=path.with_name(path.name+".tmp")
        with tmp.open("wb") as f: np.savez_compressed(f,**a)
        tmp.replace(path)
        logits,probs,y=a["logits"],a["probabilities"],a["labels"]
        print(f"Saved one full pass: {path}")
    return {"model_key":model_key,"model_name":display,"condition":name,"severity":severity,"n_images":N_TEST,"top1_accuracy":float(np.mean(logits.argmax(1)==y)),"logits_shape":list(logits.shape),"probabilities_shape":list(probs.shape),"cache_file":str(path.relative_to(OUTPUT_ROOT)),"checkpoint_sha256":base["checkpoint_sha256"]}


## 5. Cache all conditions and audit

This loop is over the two models and ten conditions only, never over the three saved splits. It reports ordinary top-1 accuracy for diagnostics; these numbers must not be used to select checkpoints or tune anything. A clean accuracy at or below 10% chance stops the notebook so checkpoint mapping can be investigated. Corruption accuracy need not decrease strictly with severity; inspect the full table for obvious anomalies without using it for model selection.


In [8]:
rows=[]
for key,(display,checkpoint,config_path) in MODEL_SPECS.items():
    model = None
    if not CACHE_REUSE_ONLY:
        model=resnet18_cifar(num_classes=10)
        model.load_state_dict(torch.load(checkpoint,map_location="cpu",weights_only=True),strict=True)
        model=model.to(DEVICE).eval()
    model_output_dir = OUTPUT_ROOT/key
    model_output_dir.mkdir(parents=True,exist_ok=True)
    metadata_path = model_output_dir/"model_metadata.json"
    if not CACHE_REUSE_ONLY:
        metadata_path.write_text(json.dumps(MODEL_META[key],indent=2),encoding="utf-8")
    print(f"\n{display}")
    for name,severity in CONDITIONS:
        row=cache_one(key,model,name,severity); rows.append(row)
        print(f"{name:16s} severity={str(severity):>4s} accuracy={row['top1_accuracy']:.4f} logits={row['logits_shape']} probs={row['probabilities_shape']}")
    if model is not None:
        del model
        gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()

summary=pd.DataFrame(rows)
if len(summary)!=20 or summary.groupby("model_key").size().to_dict()!={"augmix_resnet18":10,"standard_resnet18":10}:
    raise ValueError("Expected exactly 20 model-condition caches.")
clean_accuracy = summary.loc[summary["condition"]=="clean", "top1_accuracy"]
if CACHE_REUSE_ONLY:
    saved_summary = pd.read_csv(accuracy_path)
    keys = ["model_key", "condition", "severity"]
    summary_for_check = summary.copy()
    saved_for_check = saved_summary.copy()
    summary_for_check["severity"] = summary_for_check["severity"].fillna(-1)
    saved_for_check["severity"] = saved_for_check["severity"].fillna(-1)
    checked = summary_for_check.merge(
        saved_for_check, on=keys, suffixes=("_current", "_saved"), validate="one_to_one"
    )
    if len(checked) != 20 or not np.allclose(
        checked["top1_accuracy_current"], checked["top1_accuracy_saved"], atol=1e-12, rtol=0
    ):
        raise ValueError("Recomputed cache accuracy does not match the saved Stage 4 accuracy summary.")
    print("Saved accuracy summary matches all caches; validated files were not rewritten.")
else:
    summary.to_csv(OUTPUT_ROOT/"cache_accuracy_summary.csv",index=False)
if len(clean_accuracy)!=2 or (clean_accuracy <= 0.10).any():
    raise ValueError("A clean accuracy is at or below 10% chance; stop and inspect model/checkpoint mapping.")
if CACHE_REUSE_ONLY:
    print(f"Existing manifest retained: {manifest_path}")
else:
    manifest={"stage":"stage4_model_output_cache","n_images_per_condition":N_TEST,"split_seeds_reusing_caches":[42,43,44],"conditions":[{"condition":n,"severity":s} for n,s in CONDITIONS],"models":MODEL_META,"cache_files":summary["cache_file"].tolist(),"accuracy_summary":"cache_accuracy_summary.csv"}
    manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")



Standard ResNet-18
Saved one full pass: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions/standard_resnet18/clean_test.npz
clean            severity=None accuracy=0.9501 logits=[10000, 10] probs=[10000, 10]
Saved one full pass: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions/standard_resnet18/gaussian_noise_severity_1.npz
gaussian_noise   severity=   1 accuracy=0.7989 logits=[10000, 10] probs=[10000, 10]
Saved one full pass: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions/standard_resnet18/gaussian_noise_severity_3.npz
gaussian_noise   severity=   3 accuracy=0.4055 logits=[10000, 10] probs=[10000, 10]
Saved one full pass: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions/standard_resnet18/gaussian_noise_severity_5.npz
gaussian_noise   severity=   5 accuracy=0.2652 logits=[10000, 10] probs=[10000, 10]
Saved one full pass: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictio

3382

In [12]:
from IPython.display import display

In [13]:

# Final non-inference audit: re-open and validate every cache before review.
audit=[]
for key,meta in MODEL_META.items():
    for name,severity in CONDITIONS:
        images,labels,source_rows=condition_data(name,severity)
        base={**meta,"condition":name,"severity":severity,"n_images":N_TEST,"canonical_original_test_indices":[0,N_TEST-1],"cifar10c_source_row_range":None if name=="clean" else [int(source_rows[0]),int(source_rows[-1])+1],"transform":["ToTensor",f"Normalize(mean={CIFAR_MEAN}, std={CIFAR_STD})"],"model_eval":True,"inference_mode":True}
        path=OUTPUT_ROOT/key/("clean_test.npz" if name=="clean" else f"{name}_severity_{severity}.npz")
        with np.load(path,allow_pickle=False) as z: a={k:z[k] for k in z.files}
        logits,probs,y,stored=validate_cache(a,base,labels,source_rows)
        audit.append({"model":meta["display_name"],"condition":name,"severity":severity,"logits":logits.shape,"probabilities":probs.shape,"index_label_alignment":"PASS"})
if len(audit)!=20: raise ValueError(f"Audited {len(audit)} caches, expected 20.")
print(f"Cache files audited: {len(audit)}/20")
print("Canonical index and label alignment: PASS for all model-conditions")
display(pd.DataFrame(audit))
print("\nTop-1 accuracy summary (diagnostic only):")
display(pd.read_csv(OUTPUT_ROOT/"cache_accuracy_summary.csv"))
print(f"Cache directory: {OUTPUT_ROOT}")
print("Stop for review here; do not proceed to Stage 5 yet.")


Cache files audited: 20/20
Canonical index and label alignment: PASS for all model-conditions


,model,condition,severity,logits,probabilities,index_label_alignment
0,Standard ResNet-18,clean,NaN,"(10000, 10)","(10000, 10)",PASS
1,Standard ResNet-18,gaussian_noise,1.0,"(10000, 10)","(10000, 10)",PASS
2,Standard ResNet-18,gaussian_noise,3.0,"(10000, 10)","(10000, 10)",PASS
3,Standard ResNet-18,gaussian_noise,5.0,"(10000, 10)","(10000, 10)",PASS
4,Standard ResNet-18,defocus_blur,1.0,"(10000, 10)","(10000, 10)",PASS
5,Standard ResNet-18,defocus_blur,3.0,"(10000, 10)","(10000, 10)",PASS
6,Standard ResNet-18,defocus_blur,5.0,"(10000, 10)","(10000, 10)",PASS
7,Standard ResNet-18,brightness,1.0,"(10000, 10)","(10000, 10)",PASS
8,Standard ResNet-18,brightness,3.0,"(10000, 10)","(10000, 10)",PASS
9,Standard ResNet-18,brightness,5.0,"(10000, 10)","(10000, 10)",PASS



Top-1 accuracy summary (diagnostic only):


,model_key,model_name,condition,severity,n_images,top1_accuracy,logits_shape,probabilities_shape,cache_file,checkpoint_sha256
0,standard_resnet18,Standard ResNet-18,clean,NaN,10000,0.9501,"[10000, 10]","[10000, 10]",standard_resnet18/clean_test.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
1,standard_resnet18,Standard ResNet-18,gaussian_noise,1.0,10000,0.7989,"[10000, 10]","[10000, 10]",standard_resnet18/gaussian_noise_severity_1.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
2,standard_resnet18,Standard ResNet-18,gaussian_noise,3.0,10000,0.4055,"[10000, 10]","[10000, 10]",standard_resnet18/gaussian_noise_severity_3.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
3,standard_resnet18,Standard ResNet-18,gaussian_noise,5.0,10000,0.2652,"[10000, 10]","[10000, 10]",standard_resnet18/gaussian_noise_severity_5.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
4,standard_resnet18,Standard ResNet-18,defocus_blur,1.0,10000,0.9472,"[10000, 10]","[10000, 10]",standard_resnet18/defocus_blur_severity_1.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
5,standard_resnet18,Standard ResNet-18,defocus_blur,3.0,10000,0.8913,"[10000, 10]","[10000, 10]",standard_resnet18/defocus_blur_severity_3.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
6,standard_resnet18,Standard ResNet-18,defocus_blur,5.0,10000,0.5817,"[10000, 10]","[10000, 10]",standard_resnet18/defocus_blur_severity_5.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
7,standard_resnet18,Standard ResNet-18,brightness,1.0,10000,0.9497,"[10000, 10]","[10000, 10]",standard_resnet18/brightness_severity_1.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
8,standard_resnet18,Standard ResNet-18,brightness,3.0,10000,0.9390,"[10000, 10]","[10000, 10]",standard_resnet18/brightness_severity_3.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...
9,standard_resnet18,Standard ResNet-18,brightness,5.0,10000,0.9114,"[10000, 10]","[10000, 10]",standard_resnet18/brightness_severity_5.npz,6d0961e543a27780c58363a3136fe7a318c4b79e306ae5...


Cache directory: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions
Stop for review here; do not proceed to Stage 5 yet.


In [14]:
manifest = json.loads((OUTPUT_ROOT / 'cache_manifest.json').read_text(encoding='utf-8'))
summary = pd.read_csv(OUTPUT_ROOT / 'cache_accuracy_summary.csv')
missing = [name for name in manifest['cache_files'] if not (OUTPUT_ROOT / name).is_file()]
metadata_files = [OUTPUT_ROOT / key / 'model_metadata.json' for key in MODEL_META]
assert manifest['stage'] == 'stage4_model_output_cache'
assert len(manifest['cache_files']) == 20 and len(summary) == 20
assert not missing and all(path.is_file() for path in metadata_files)
print('Manifest cache entries:', len(manifest['cache_files']))
print('Accuracy summary rows:', len(summary))
print('Missing cache files:', missing)
print('Model metadata files:', [path.name for path in metadata_files])
manifest=json.loads((OUTPUT_ROOT/'cache_manifest.json').read_text(encoding='utf-8')); summary=pd.read_csv(OUTPUT_ROOT/'cache_accuracy_summary.csv'); missing=[name for name in manifest['cache_files'] if not (OUTPUT_ROOT/name).is_file()]; metadata_files=[OUTPUT_ROOT/key/'model_metadata.json' for key in MODEL_META]; assert manifest['stage']=='stage4_model_output_cache'; assert len(manifest['cache_files'])==20 and len(summary)==20; assert not missing and all(path.is_file() for path in metadata_files); print('Manifest cache entries:',len(manifest['cache_files']),'| Accuracy summary rows:',len(summary),'| Missing cache files:',missing,'| Model metadata files:',[str(path.relative_to(OUTPUT_ROOT)) for path in metadata_files],'| Drive output audit: PASS')

Manifest cache entries: 20
Accuracy summary rows: 20
Missing cache files: []
Model metadata files: ['model_metadata.json', 'model_metadata.json']
Manifest cache entries: 20 | Accuracy summary rows: 20 | Missing cache files: [] | Model metadata files: ['standard_resnet18/model_metadata.json', 'augmix_resnet18/model_metadata.json'] | Drive output audit: PASS
